# Pre- vs. post-SHINE cohort comparison: similarity to the semantic hierarchy

Are the pre-SHINE and post-SHINE cohorts *equally similar to the Kiani hierarchy* encoded in the curated image paths (`animate/animal/body/bird/...`)?<br>
Compares each cohort's behavioral RDM, and the hierarchy implied by its MDS geometry, with the directory tree, and tests whether the two cohorts differ, for **retained subjects only** (production cohort, `status == "full data"`, `task_version >= 4.0`).<br>
Companion notebooks:
- `0_sensory.ipynb` - does SHINE change the stimuli themselves?
- `1_demographics_and_behavior.ipynb` - do they behave the same?
- `2_data_quality.ipynb` - is the data equally good?
- `3_geometry.ipynb` - do they give the same similarity structure?
- `5_ground_truth.ipynb` - are they equally similar to earlier ratings and recordings?
- `6_decomposition.ipynb` - what is the similarity made of: semantic and sensory weights?

Backend helpers live in `reference_agreement.py` (RDM-level) and `tree_agreement.py` (tree-level).

**Statistics.** The reference RDM `sem_km` (tree-edge distance) takes a handful of integer values, so it is heavily tied: **Kendall tau-a is the primary statistic and Spearman rho the secondary one.** Significance comes from an image-relabeling permutation test restricted to the pairs a cohort observed; uncertainty from a participant bootstrap. Each cohort's RDM is `build_behavioral_rdm(..., aggregation="simple")`, a condensed vector with NaN for pairs no retained subject judged. Expensive steps are cached under `analysis/results/pre_vs_post_shine/` and reused only under a matching fingerprint of subjects, n and seed.

In [1]:
import sys
from pathlib import Path



import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s1_pre_vs_post_shine/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from analysis.utils.stats import mwu_effect_size, print_comparison_line
from analysis.utils.equivalence import (
    equivalence_sensitivity, equivalence_test_rank_biserial, print_equivalence_line, resolvable_bound,
)
from analysis.utils.plotting import fig_equivalence_forest
from analysis.s0_build_rdms.common import load_rdm
from analysis.s0_build_rdms.behavior.spam_ratings import build_behavioral_rdm
from analysis.s1_pre_vs_post_shine.demographics_and_behavior import COHORTS, COHORT_LABEL, load_retained_cohorts
from analysis.s1_pre_vs_post_shine.geometry import MDS_CACHE_DIR, parse_subject_trials
from analysis.s1_pre_vs_post_shine.reference_agreement import (
    REFERENCE_CACHE_DIR, bh_within_cohort, bootstrap_cohort_difference, bootstrap_combined_fit_multi,
    cohort_difference_equivalence, fig_agreement_with_ci, fig_bootstrap_distributions, fig_fit_vs_lower_bound,
    loo_model_fit, observed_cohort_fit, percentile_ci, split_by_cohort,
)
from analysis.s1_pre_vs_post_shine.tree_agreement import (
    MDS_DIMS, agglomerative_agreement, dendrogram_shuffle_test, fig_baker_gamma, fig_hdbscan_agreement,
    hdbscan_level_agreement, manifest_parts, path_tree_to_linkage, perceptual_embeddings, perceptual_tree,
)
from analysis.utils.stats import label_shuffle_test
from SpAM_Simulations.measures.cluster_stability import cophenetic_ranks

import plotly.io as pio
pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

## 0. Load data

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
rdm = {c: build_behavioral_rdm(f"{c}_shine", cache_dir=cache_dir, aggregation="simple") for c in COHORTS}
km = load_rdm("sem_km")

cohorts = load_retained_cohorts(cache_dir)
subject_ids = {c: cohorts["participant_ids"][c].tolist() for c in COHORTS}
trials_by_subject, n_conds = parse_subject_trials(cohorts["df_trials"], subject_ids)
trials = split_by_cohort(trials_by_subject, subject_ids)
parts = manifest_parts()

SEED, N_PERM, N_BOOT, N_BOOT_DIFF = 0, 1000, 500, 1000
cache = lambda name: REFERENCE_CACHE_DIR / name
pd.DataFrame({
    COHORT_LABEL[c]: {"retained subjects": len(subject_ids[c]), "pairs rated (of 262,450)": int((~np.isnan(rdm[c])).sum())}
    for c in COHORTS
})

[behavioral] variant='pre_shine': 79 participants.
[behavioral] Built 1738 per-trial partial observation sets (screening + experimental pooled).
[behavioral] Pair coverage: 64.57% covered (92977 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.95, sd=1.13.
[behavioral] WARNING: 92977 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_pre_simple.npy  (length 262450)
[behavioral] variant='post_shine': 76 participants.
[behavioral] Built 1672 per-trial partial observation sets (screening + experimental pooled).
[behavioral] Pair coverage: 63.09% covered (96882 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.92, sd=1.11.
[behavioral] WARNING: 96882 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_post_simple.npy  (length 26

,Pre-SHINE,Post-SHINE
retained subjects,79,76
"pairs rated (of 262,450)",169473,165568


## 1. Does each cohort's RDM resemble the Kiani tree?

Per cohort: tau-a (primary) and rho (secondary) between the behavioral RDM and `sem_km` on the pairs the cohort rated; a one-sided image-relabeling permutation p (the reference's image identity is shuffled, rows and columns together, which keeps its internal dependence); and a 95% participant-bootstrap CI of the combined RDM's tau-a (participants resampled with replacement, RDM re-aggregated each draw).

In [3]:
agreement = {}
for c in COHORTS:
    permutation = label_shuffle_test(
        rdm[c], km, n_perm=N_PERM, seed=SEED, cache_path=cache(f"km_permutation_{c}_n{N_PERM}_seed{SEED}.npz"),
    )
    boots = bootstrap_combined_fit_multi(
        trials[c], {"km": km}, n_conds, n_boot=N_BOOT, seed=SEED, cache_path=cache(f"km_bootstrap_{c}_n{N_BOOT}_seed{SEED}.pkl"),
    )
    agreement[c] = {"permutation": permutation, "boots": boots, "ci": percentile_ci(boots["kendall_a"])}

summary = pd.DataFrame([{
    "cohort": c, "target": "km", "tau_a": a["permutation"].observed["kendall_a"], "ci_lo": a["ci"][0], "ci_hi": a["ci"][1],
    "spearman": a["permutation"].observed["spearman"], "permutation_p (tau-a)": a["permutation"].p_value["kendall_a"],
} for c, a in agreement.items()])
summary.round(4)

,cohort,target,tau_a,ci_lo,ci_hi,spearman,permutation_p (tau-a)
0,pre,km,0.2094,0.1874,0.2126,0.3176,0.001
1,post,km,0.2100,0.1868,0.2162,0.3196,0.001


In [4]:
fig_agreement_with_ci(summary, targets={"km": "Kiani tree (sem_km)"}).show()
fig_bootstrap_distributions({c: agreement[c]["boots"] for c in COHORTS}, "kendall_a", x_title="tau-a of the combined RDM vs. sem_km (participant bootstrap)").show()

### 1.1 Per-participant fit against the lower noise bound

Each participant's own RDM covers only a small share of pairs, so single-participant fits are low. The lower noise bound is the same participant's agreement with the combined RDM of everyone else (leave-one-participant-out, Nili et al. 2014): a reference at or above it explains what one participant shares with the group.

In [5]:
loo = {c: loo_model_fit(trials[c], km, n_conds, cache_path=cache(f"km_loo_{c}.pkl")) for c in COHORTS}
fig_fit_vs_lower_bound(loo).show()
pd.DataFrame({
    COHORT_LABEL[c]: {
        "participants": len(loo[c]),
        "mean tau-a vs sem_km": loo[c]["model_kendall_a"].mean(),
        "mean tau-a lower bound": loo[c]["lower_kendall_a"].mean(),
        "Wilcoxon p (reference vs bound)": wilcoxon(loo[c]["model_kendall_a"].dropna(), loo[c]["lower_kendall_a"].dropna()).pvalue,
    } for c in COHORTS
}).T.round(4)

,participants,mean tau-a vs sem_km,mean tau-a lower bound,Wilcoxon p (reference vs bound)
Pre-SHINE,79.0,0.1856,0.1306,0.0
Post-SHINE,76.0,0.1888,0.1390,0.0


## 2. Are the cohorts equally similar to the tree?

A difference test cannot show that two cohorts are alike, so equivalence is tested directly (TOST in its 90%-CI form, as in notebooks 1-3). There is no pre-defined smallest effect size of interest, so each bound is set by what the design can resolve (`analysis/utils/equivalence.py`), not by a judgement that a difference is negligible.

**(a) Per-participant fit**, as in notebooks 1-2: rank-biserial effect of the pre vs post per-participant tau-a (independent samples, bootstrap CI); the bound is the smallest at which a true difference of zero is declared equivalent with 80% probability (simulated).

**(b) Combined-RDM tau-a**: D = tau-a(pre) - tau-a(post). The cohorts are disjoint participants, so each is bootstrapped on its own and re-aggregated per draw; D must have its 90% CI inside +/- bound. The bound is (z(0.95) + z(0.90)) x SE (`tost_bound_multiplier`), rounded up to 0.01: the smallest at which a true difference of zero is declared equivalent with 80% probability, the same rule as the simulated bound in (a) (checked by calibration in `5_ground_truth.ipynb`, section 4). Scored on each cohort's own observed pairs (`own`) and on only the pairs both cohorts observed (`joint`), which removes any coverage difference.

In [6]:
fit = {c: loo[c]["model_kendall_a"].dropna().to_numpy() for c in COHORTS}
print_comparison_line(mwu_effect_size(fit["pre"], fit["post"], label_a="pre", label_b="post"))
fit_bound = resolvable_bound(equivalence_sensitivity(len(fit["pre"]), len(fit["post"])))
fit_equivalence = equivalence_test_rank_biserial(fit["pre"], fit["post"], bound=fit_bound, label_a="pre", label_b="post")
print_equivalence_line(fit_equivalence)
fig_equivalence_forest({"per-participant tau-a vs sem_km": fit_equivalence}, x_title="rank-biserial r (pre vs post, 90% CI)",
                       title="Per-participant similarity to the Kiani tree").show()

  pre vs post: U=2997.0  p=0.9871 (not significant)  r=0.002
  pre vs post: effect=0.002  90% CI [-0.155, 0.156]  bound=+/-0.3 (equivalent)


In [7]:
joint = ~np.isnan(rdm["pre"]) & ~np.isnan(rdm["post"])
boots_diff = bootstrap_cohort_difference(
    trials["pre"], trials["post"], {"km": km}, n_conds, n_boot=N_BOOT_DIFF, seed=SEED, pair_masks={"own": None, "joint": joint},
    cache_path=cache(f"km_cohort_difference_n{N_BOOT_DIFF}_seed{SEED}.pkl"),
)
observed = {
    "own": observed_cohort_fit(rdm["pre"], rdm["post"], {"km": km}),
    "joint": observed_cohort_fit(rdm["pre"], rdm["post"], {"km": km}, mask=joint),
}
difference = cohort_difference_equivalence(observed, boots_diff, len(subject_ids["pre"]), len(subject_ids["post"]))
print(f"pairs observed: pre {int((~np.isnan(rdm['pre'])).sum()):,}, post {int((~np.isnan(rdm['post'])).sum()):,}, both {int(joint.sum()):,}")
difference[["pairs", "statistic", "effect", "ci_low", "ci_high", "bound", "min_bound", "se", "equivalent"]].round(4)

pairs observed: pre 169,473, post 165,568, both 107,040


,pairs,statistic,effect,ci_low,ci_high,bound,min_bound,se,equivalent
0,joint,kendall_a,0.0036,-0.0148,0.0193,0.04,0.0193,0.0103,True
1,joint,spearman,0.0038,-0.0234,0.0274,0.05,0.0274,0.0154,True
2,own,kendall_a,-0.0005,-0.0178,0.0159,0.03,0.0178,0.0102,True
3,own,spearman,-0.0020,-0.0276,0.0227,0.05,0.0276,0.0152,True


In [8]:
fig_equivalence_forest(
    {f"{'tau-a' if r.statistic == 'kendall_a' else 'rho'}, {r.pairs} pairs": r._asdict() for r in difference.itertuples()},
    x_title="pre - post (90% CI; grey ticks = equivalence bound)", title="Equivalence of pre- and post-SHINE similarity to the Kiani tree",
).show()

## 3. Does the geometry imply the same hierarchy?

The exact hierarchy is the directory tree itself, converted to a dendrogram (`path_tree_to_linkage`; each directory is a merge point, so cophenetic distance falls with shared depth). The perceptual side is each cohort's cached SMACOF embedding (`3_geometry.ipynb`'s fits, `MDS_CACHE_DIR`) at d = 3, 9, 15 dimensions, clustered with **average linkage**: the v5 simulation report (`SpAM_Simulations/sim_results/v5/report/`) finds average outperforms ward and complete throughout (Baker's gamma 0.611 vs 0.579 and 0.471).

Agreement is **Baker's gamma**, the rank correlation of the two dendrograms' cophenetic distances (k-free, so it summarises the whole merge order), against a null that shuffles image identity in the tree. It is dominated by coarse structure, since most pairs merge near the root. *Cophenetic fidelity* is how well each dendrogram keeps its own embedded geometry, which qualifies the gamma. For pre vs post this is descriptive: a participant bootstrap would need an MDS refit per draw (about 10 h).

In [9]:
z_km = path_tree_to_linkage(parts)
embedded = {c: perceptual_embeddings(rdm[c], c, MDS_DIMS, MDS_CACHE_DIR) for c in COHORTS}

rows = []
for c in COHORTS:
    table = agglomerative_agreement(z_km, embedded[c]).set_index("ndim")
    for d in MDS_DIMS:
        shuffle = dendrogram_shuffle_test(
            cophenetic_ranks(perceptual_tree(embedded[c][d])), parts, n_perm=N_PERM, seed=SEED,
            cache_path=cache(f"km_gamma_shuffle_{c}_d{d}_n{N_PERM}_seed{SEED}.npz"),
        )
        rows.append({"cohort": c, "ndim": d, "cophenetic_fidelity": table.at[d, "cophenetic_fidelity"], **shuffle.summary()})
gamma = pd.DataFrame(rows)
fig_baker_gamma(gamma, title="Baker's gamma vs. the Kiani tree (average linkage; grey = 95th percentile of the shuffle null)").show()
gamma.round(4)

,cohort,ndim,cophenetic_fidelity,baker_gamma,null_mean,null_sd,null_p95,p_value
0,pre,3,0.6355,0.9813,-0.0001,0.0027,0.0053,0.001
1,pre,9,0.6696,0.8039,0.0000,0.0040,0.0072,0.001
2,pre,15,0.7009,0.8070,0.0000,0.0040,0.0073,0.001
3,post,3,0.6606,0.6821,0.0001,0.0039,0.0073,0.001
4,post,9,0.7146,0.6731,0.0001,0.0039,0.0069,0.001
5,post,15,0.7461,0.6857,0.0001,0.0038,0.0070,0.001


### 3.1 HDBSCAN against KM categories

HDBSCAN labels isolated images as noise instead of forcing them into a cluster, so it complements the average-linkage tree. It yields flat labels, not a dendrogram, so each labelling is scored against the KM categories at each directory level by the adjusted Rand index over the images it clustered (`frac_noise` and `frac_shared` say how many were left out). Descriptive only; run on the same embeddings.

In [10]:
hdbscan = pd.concat(
    [hdbscan_level_agreement(embedded[c], parts, (1, 2, 3)).assign(cohort=c) for c in COHORTS], ignore_index=True,
)
fig_hdbscan_agreement(hdbscan, level_names={1: "Level 1: domain", 2: "Level 2: mid-level", 3: "Level 3: basic"}, ndim=9).show()
hdbscan[hdbscan["ndim"] == 9].pivot_table(index=["cohort", "min_cluster_size"], columns="km_level", values=["ari", "frac_noise"]).round(2)

ari             frac_noise            
km_level                    1     2     3          1     2     3
cohort min_cluster_size                                         
post   2                 0.21  0.52  0.67       0.17  0.17  0.17
       3                 0.81  0.60  0.21       0.06  0.06  0.06
       5                 0.47  0.95  0.44       0.12  0.12  0.12
       10                0.50  0.99  0.45       0.12  0.12  0.12
       20                0.52  0.99  0.53       0.29  0.29  0.29
pre    2                 0.19  0.48  0.59       0.21  0.21  0.21
       3                 0.42  0.88  0.49       0.09  0.09  0.09
       5                 0.45  0.92  0.46       0.11  0.11  0.11
       10                0.46  0.94  0.49       0.13  0.13  0.13
       20                0.46  0.93  0.51       0.15  0.15  0.15

## 4. Multiple comparisons

Benjamini-Hochberg within each cohort over its confirmatory family: the tau-a permutation test and the three Baker's-gamma shuffle tests (d = 3, 9, 15). Cohorts are never pooled, since pre and post are distinct claims. Spearman rho, the bootstrap CIs and the equivalence tests are not part of the family.

In [11]:
family = pd.concat([
    pd.DataFrame({"cohort": c, "test": "tau-a permutation", "p_value": [agreement[c]["permutation"].p_value["kendall_a"]]})
    for c in COHORTS
] + [gamma.assign(test=lambda d: "Baker gamma, d = " + d["ndim"].astype(str))[["cohort", "test", "p_value"]]], ignore_index=True)
bh_within_cohort(family).round(4)

,cohort,test,p_value,p_fdr
0,pre,tau-a permutation,0.001,0.001
1,post,tau-a permutation,0.001,0.001
2,pre,"Baker gamma, d = 3",0.001,0.001
3,pre,"Baker gamma, d = 9",0.001,0.001
4,pre,"Baker gamma, d = 15",0.001,0.001
5,post,"Baker gamma, d = 3",0.001,0.001
6,post,"Baker gamma, d = 9",0.001,0.001
7,post,"Baker gamma, d = 15",0.001,0.001
